# DS5 — Feature Extraction

**Dataset:** Dataset 5 — German adolescents (RCT)  
**Stage:** 1 — Dataset-specific feature extraction (no cross-dataset harmonization)  
**Input files:** `outputs/cleaned/ds5_*.csv`  
**Output files:** `outputs/features/ds5/`

---

## Outputs produced

| File | Rows | Contents |
|---|---|---|
| `ds5_behavior_features.csv` | ~573 | All behavioral participants + study_condition |
| `ds5_survey_features.csv` | ~572 | Participants with pre-survey + behavioral |
| `ds5_prepost_features.csv` | ~284 | Participants with behavioral + both surveys |

---

## Event definitions

- **What counts as an event row:** Every row in `ds5_behavioral_clean.csv` (one friction event per row).
- **Intervention exposure:** `is_intervention_event == 1` (friction was shown; duration > 0)
- **Reintervention events:** `is_reintervention == 1` (subset; reintervention condition only)
- **Outcome-bearing rows:** `event_outcome_simple` ∈ {opened, dismissed, closed}  
  Excluded: 'missing' (3,457 rows, 1.2%) and 'dismissed_other' (119 rows, 0.04%)
- **Study condition:** NOT in behavioral file — joined from pre-survey.
  Values: `control` / `default` / `reintervention`

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("__file__").resolve().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd

CLEANED = PROJECT_ROOT / "outputs" / "cleaned"
OUT = PROJECT_ROOT / "outputs" / "features" / "ds5"
OUT.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Output folder:", OUT)

## 1. Load cleaned files

In [ ]:
from src.features.ds5_features import load_ds5_files

beh, pre, post, linkage = load_ds5_files(CLEANED)

print(f"Behavioral rows:         {len(beh):,}")
print(f"Behavioral participants:  {beh['participant_id_canonical'].nunique():,}")
print(f"Pre-survey rows:         {len(pre):,}")
print(f"Post-survey rows:        {len(post):,}")
print(f"Linkage rows:            {len(linkage):,}")

# Condition balance
print("\nCondition distribution (pre-survey):")
print(pre['cond'].value_counts())

In [ ]:
# Linkage overlap
beh_ids = set(beh['participant_id_canonical'].unique())
pre_ids = set(pre['participant_id_canonical'].unique())
post_ids = set(post['participant_id_canonical'].unique())
print(f"Behavioral + pre-survey:          {len(beh_ids & pre_ids):,}")
print(f"Behavioral + both surveys:        {len(beh_ids & pre_ids & post_ids):,}")
print(f"Behavioral only (no pre-survey):  {len(beh_ids - pre_ids):,}")

## 2. Build behavior features

Feature groups:
- **G1** Volume and activity (DIRECT, High) — includes study_condition
- **G2** Outcome distribution (DIRECT, High)
- **G3** Intervention event outcome rate (DIRECT, High)
- **G4** Intervention duration (DIRECT/PROXY, Medium)
- **G5** Reintervention features (DIRECT, Medium-High — DS5+DS6 unique)
- **G6** App structure (DIRECT, High)
- **G7** Temporal structure (DIRECT-DERIVED, High)
- **G8** Longitudinal adaptation (PROXY, Medium)
- **G9** Override tendency (PROXY, Medium)

In [ ]:
from src.features.ds5_features import build_ds5_behavior_features

print("Building behavioral features...")
beh_feats = build_ds5_behavior_features(beh, pre)
print(f"Shape: {beh_feats.shape}")
beh_feats.head(3)

In [ ]:
# Condition coverage
print("Condition distribution in behavior features:")
print(beh_feats['study_condition'].value_counts(dropna=False))
print(f"\nParticipants with NaN condition: {beh_feats['study_condition'].isna().sum()}")
# (1 behavioral participant has no pre-survey record)

In [ ]:
print("Key outcome distributions by condition:")
print(beh_feats.groupby('study_condition')[['rate_opened', 'rate_dismissed', 'rate_closed']].mean().round(3))

In [ ]:
print("Reintervention feature coverage:")
print(beh_feats.groupby('study_condition')[['has_reintervention', 'mean_reintervention_delay_sec']].mean().round(3))
print()
print("Reintervention open rate (meaningful for reintervention condition only):")
print(beh_feats.groupby('study_condition')['reintervention_open_rate'].mean().round(3))

In [ ]:
out_path = OUT / "ds5_behavior_features.csv"
beh_feats.to_csv(out_path, index=False)
print(f"Saved: {out_path}  ({len(beh_feats):,} rows, {beh_feats.shape[1]} columns)")

## 3. Build pre-survey features

Feature groups:
- **G10** Pre-survey psychosocial scales: belonging, ostracism, FoMO (DIRECT, High)  
  Available for: ~572 participants with behavioral + pre-survey

In [ ]:
from src.features.ds5_features import build_ds5_survey_features

survey_feats = build_ds5_survey_features(beh, pre)
print(f"Shape: {survey_feats.shape}")

print("\nScale distribution by condition:")
scale_cols = ['belong_mean_pre', 'ostracism_mean_pre', 'fomo_mean_pre', 'digital_wellbeing_pre', 'prob_self_pre']
print(survey_feats.groupby('study_condition')[scale_cols].mean().round(3))

In [ ]:
out_path = OUT / "ds5_survey_features.csv"
survey_feats.to_csv(out_path, index=False)
print(f"Saved: {out_path}  ({len(survey_feats):,} rows, {survey_feats.shape[1]} columns)")

## 4. Build pre-post change features

Feature groups:
- **G11** Post-survey psychosocial (DIRECT, High)  
- **G12** Pre-post signed change scores (DIRECT-DERIVED, Medium)  
Available for: ~284 participants with behavioral + both surveys

In [ ]:
from src.features.ds5_features import build_ds5_prepost_features

prepost_feats = build_ds5_prepost_features(beh, pre, post)
print(f"Shape: {prepost_feats.shape}")
print("\nChange scores by condition (mean ± note interpretation within condition):")
change_cols = [c for c in prepost_feats.columns if '_change' in c]
print(prepost_feats.groupby('study_condition')[change_cols].mean().round(3))

In [ ]:
out_path = OUT / "ds5_prepost_features.csv"
prepost_feats.to_csv(out_path, index=False)
print(f"Saved: {out_path}  ({len(prepost_feats):,} rows, {prepost_feats.shape[1]} columns)")

## 5. Output summary

In [ ]:
print("=" * 60)
print("DS5 Feature Extraction — Output Summary")
print("=" * 60)

for fname, df in [
    ('ds5_behavior_features.csv', beh_feats),
    ('ds5_survey_features.csv', survey_feats),
    ('ds5_prepost_features.csv', prepost_feats),
]:
    n_features = df.shape[1] - 1
    print(f"  {fname}")
    print(f"    {len(df):,} participants × {n_features} features")
    missing_any = df.drop(columns='participant_id_canonical').isna().any(axis=1).sum()
    print(f"    Participants with any NaN feature: {missing_any:,}")
    print()